# Data Exploration

Run after `scripts/01_prepare_split.py`. Requires `data/manifest.csv`.

In [ ]:
import os, sys
from pathlib import Path

_here = Path.cwd()
while not (_here / 'pyproject.toml').exists() and _here != _here.parent:
    _here = _here.parent
PROJECT_ROOT = _here.resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk

df = pd.read_csv(PROJECT_ROOT / 'data' / 'manifest.csv')
# Make image/mask paths absolute so later cells don't depend on CWD
df['image'] = df['image'].apply(lambda p: str(PROJECT_ROOT / p))
df['mask']  = df['mask'].apply(lambda p: str(PROJECT_ROOT / p))

print(f'Project root: {PROJECT_ROOT}')
print(f'Total rows: {len(df)}')
print(df.head())

In [ ]:
# Split distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Patients per split
pat_counts = df.drop_duplicates('patient_id').groupby('split').size()
axes[0].bar(pat_counts.index, pat_counts.values, color=['steelblue','orange','green'])
axes[0].set_title('Patients per split')
axes[0].set_ylabel('Count')

# Quality distribution
qual_split = df.drop_duplicates('patient_id').groupby(['split','quality']).size().unstack(fill_value=0)
qual_split.T.plot(kind='bar', ax=axes[1])
axes[1].set_title('Quality by split')
axes[1].tick_params(axis='x', rotation=0)

# Images per view x phase
vp = df.groupby(['view','phase']).size().unstack()
vp.plot(kind='bar', ax=axes[2])
axes[2].set_title('Images by view/phase')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Sample overlays — 6 random training cases
sample = df[df['split'] == 'train'].sample(6, random_state=42)

cmap = plt.get_cmap('tab10')
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for col, (_, row) in enumerate(sample.iterrows()):
    img_arr = sitk.GetArrayFromImage(sitk.ReadImage(row['image']))
    gt_arr  = sitk.GetArrayFromImage(sitk.ReadImage(row['mask']))
    if img_arr.ndim == 3:
        img_arr, gt_arr = img_arr[0], gt_arr[0]
    img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-8)

    axes[0, col].imshow(img_arr, cmap='gray')
    axes[0, col].set_title(f"{row['patient_id']}\n{row['view']} {row['phase']} ({row['quality']})", fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(img_arr, cmap='gray')
    axes[1, col].imshow(gt_arr, alpha=0.4, cmap=cmap, vmin=0, vmax=3)
    axes[1, col].axis('off')

patches = [mpatches.Patch(color=cmap(i), label=l) for i, l in enumerate(['bg','LV','MYO','LA'])]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9)
plt.suptitle('Top: image  |  Bottom: image + GT mask', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Image statistics: pixel spacing and image size
spacings, heights, widths = [], [], []
for _, row in df[df['split'] == 'train'].iterrows():
    img = sitk.ReadImage(row['image'])
    sp = img.GetSpacing()
    sz = img.GetSize()
    spacings.append((sp[0], sp[1]))
    widths.append(sz[0])
    heights.append(sz[1])

spacings = np.array(spacings)
print(f'Pixel spacing (mm) — x: {spacings[:,0].min():.3f}–{spacings[:,0].max():.3f}  '
      f'y: {spacings[:,1].min():.3f}–{spacings[:,1].max():.3f}')
print(f'Image size — W: {min(widths)}–{max(widths)}  H: {min(heights)}–{max(heights)}')

In [ ]:
# Image statistics: pixel intensity histograms and size distributions
train_rows = df[df['split'] == 'train']

intensities, spacings_x, spacings_y, widths, heights = [], [], [], [], []

for _, row in train_rows.iterrows():
    img = sitk.ReadImage(row['image'])
    arr = sitk.GetArrayFromImage(img).astype(float)
    if arr.ndim == 3:
        arr = arr[0]
    sp = img.GetSpacing()
    sz = img.GetSize()
    intensities.append(arr.flatten())
    spacings_x.append(sp[0])
    spacings_y.append(sp[1])
    widths.append(sz[0])
    heights.append(sz[1])

all_px = np.concatenate(intensities)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].hist(all_px, bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Pixel intensity distribution (train)')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Count')

axes[1].hist(spacings_x, bins=30, color='darkorange', edgecolor='none')
axes[1].set_title('Pixel spacing — x (mm)')
axes[1].set_xlabel('mm / pixel')

axes[2].hist(widths, bins=20, color='forestgreen', edgecolor='none')
axes[2].set_title('Image width (pixels)')
axes[2].set_xlabel('Width')

axes[3].hist(heights, bins=20, color='mediumpurple', edgecolor='none')
axes[3].set_title('Image height (pixels)')
axes[3].set_xlabel('Height')

plt.suptitle('Training set image statistics', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Spacing x: {min(spacings_x):.3f}–{max(spacings_x):.3f} mm  '
      f'y: {min(spacings_y):.3f}–{max(spacings_y):.3f} mm')
print(f'Size W: {min(widths)}–{max(widths)}  H: {min(heights)}–{max(heights)}')